In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import sys
import os
import matplotlib.pyplot as plt
import time

sys.path.append(os.path.abspath('../common'))
from DO_structs import *
from DO_graphics import *
from DO_server_graphics import *
from DO_searching import *

sys.path.append(os.path.abspath('../planners_library/'))
from sipp import *

%matplotlib inline

# Simple Arena Task

In [41]:
# 1. Map (grid with static obstacles)
task_map = Map()
task_map.read_from_movingai_file("../maps/arena.map")

# 2. Control Set (for dynamic environment) 
env_cs = ControlSet()
env_cs.load_primitives("../data/control_set.txt")

# 3. Environmnet (both static and dynamic obstacles)
R = 1  # Ego-robot radius
environment = DynamicEnvironment(task_map, R)
environment.load_obstacles_from_dir("../dynamic_obstacles/arena_random/dynamic_obstacle_*.txt", env_cs, max_items=50)
environment.compile_safe_intervals()

In [42]:
# Ego-robot control set of primitives:
extended_cs = ControlSet()
extended_cs.load_primitives("../data/extended_set.txt")  # use extended one, for example

In [43]:
# Test case (start and goal states on the task_map)
start = DiscreteState(6, 11, 7)
goal = DiscreteState(42, 34, 11)
search_space = SIPPSearchSpace(start, 0.0, goal, extended_cs, environment, R=0, A=0, position_only=True)

# Execute the SIPP (A*) search algorithm
success, path, steps, cost, ast = astar(search_space)
success, steps, cost

(True, 8610, np.float64(587.086373))

In [44]:
# Instantiate the ego-robot as a dynamic obstacle (as it shares the same underlying trajectory representation)
robot = DynamicObstacle(extended_cs).set_parametres(R, start.i, start.j, start.theta, path)

In [45]:
# Launch high-performance parallel MP4 rendering utilizing 50 available server cores
render_simulation_parallel(
    output_file="../media/demo-arena-1.mp4", # Set the destination path; switch the extension to '.gif' if an animated GIF is preferred instead
    task_map=task_map,
    obstacle_set=environment.get_dynamic_obstacles_set(),
    robot=robot, # Pass the target scheduled robot trajectory instance here
    max_time=600.0,
    dt=2.0,
    fps=24,
    show_path=True,
    obs_line_width=0.0,
    fast_draw=False,
    dpi=200, # High-quality rendering layout: 150-200 DPI ensures crisp details while keeping the MP4 file size lightweight
    scale=2,
    workers=50  # Concurrency control: limit the CPU worker pool size to avoid resource starvation on shared clusters
)

🚀 Starting parallel rendering...
📊 Total frames: 301 | CPU Workers: 50


Rendering frames:   0%|          | 0/301 [00:00<?, ?it/s]

🎬 Assembling media file (Streaming Mode + HEVC)...


Saving MP4:   0%|          | 0/301 [00:00<?, ?it/s]

x265 [info]: HEVC encoder version 3.5+1-f0c1022b6
x265 [info]: build info [Linux][GCC 8.3.0][64 bit] 8bit+10bit+12bit
x265 [info]: using cpu capabilities: MMX2 SSE2Fast LZCNT SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
x265 [info]: Main profile, Level-6 (Main tier)
x265 [info]: Thread pool created using 64 threads
x265 [info]: Thread pool created using 64 threads
x265 [info]: Slices                              : 1
x265 [info]: frame threads / pool features       : 6 / wpp(51 rows)
x265 [info]: Coding QT: max CU size, min CU size : 64 / 8
x265 [info]: Residual QT: max TU size, max depth : 32 / 1 inter / 1 intra
x265 [info]: ME / range / subpel / merge         : hex / 57 / 2 / 3
x265 [info]: Keyframe min / max / scenecut / bias  : 24 / 250 / 40 / 5.00 
x265 [info]: Lookahead / bframes / badapt        : 20 / 4 / 2
x265 [info]: b-pyramid / weightp / weightb       : 1 / 1 / 0
x265 [info]: References / ref-limit  cu / depth  : 3 / off / on
x265 [info]: AQ: mode / str / qg-size / cu-tree  : 2 / 1.0 / 32

✅ Success! Render saved to: ../media/demo-arena-1.mp4
🧹 Cleaning up temporary frames...


# Comparative Benchmark: Baseline vs. Extended Control Sets

This test evaluates the performance of the **Safe Interval Path Planning (SIPP)** algorithm on a single test case using two different control sets: a standard baseline set (`env_cs`) and an enhanced, more expressive extended set (`extended_cs`). 

Orientation constraints are ignored by setting `position_only=True`. Since the extended control set offers higher maneuverability, it is expected to yield a more optimal trajectory with lower path cost.

In [16]:
# Define a unified test case (shared start and goal configurations)
start = DiscreteState(39, 5, 0)
goal = DiscreteState(3, 39, 0)

# Solve using the Extended Control Set (higher expressiveness)
search_space_1 = SIPPSearchSpace(start, 0.0, goal, extended_cs, environment, R=0, A=0, position_only=True)
_, path_1, _, c1, _ = astar(search_space_1)

# Solve using the Baseline Control Set (standard primitives)
search_space_2 = SIPPSearchSpace(start, 0.0, goal, env_cs, environment, R=0, A=0, position_only=True)
_, path_2, _, c2, _  = astar(search_space_2)

print(f"Costs: {c1} (extended) vs {c2} (basic)")

Costs: 580.75297 (extended) vs 687.394832 (basic)


In [17]:
# Construct robots into DynamicObstacle objects for rendering
robot_1 = DynamicObstacle(extended_cs).set_parametres(R, start.i, start.j, start.theta, path_1)
robot_2 = DynamicObstacle(env_cs).set_parametres(R, start.i, start.j, start.theta, path_2)

In [18]:
# Generate parallel simulations for both trajectories
common_kwargs = dict(
    task_map=task_map, obstacle_set=environment.get_dynamic_obstacles_set(),
    max_time=700.0, dt=2.0, fps=24, show_path=True, obs_line_width=0.0,
    fast_draw=False, dpi=200, scale=2, workers=50
)

# Render Extended Control Set simulation
render_simulation_parallel(output_file="../media/demo-arena-2-extended.mp4", robot=robot_1, **common_kwargs)

# Render Baseline Control Set simulation
render_simulation_parallel(output_file="../media/demo-arena-2-basic.mp4", robot=robot_2, **common_kwargs)

🚀 Starting parallel rendering...
📊 Total frames: 351 | CPU Workers: 50


Rendering frames:   0%|          | 0/351 [00:00<?, ?it/s]

🎬 Assembling media file (Streaming Mode + HEVC)...


Saving MP4:   0%|          | 0/351 [00:00<?, ?it/s]

x265 [info]: HEVC encoder version 3.5+1-f0c1022b6
x265 [info]: build info [Linux][GCC 8.3.0][64 bit] 8bit+10bit+12bit
x265 [info]: using cpu capabilities: MMX2 SSE2Fast LZCNT SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
x265 [info]: Main profile, Level-6 (Main tier)
x265 [info]: Thread pool created using 64 threads
x265 [info]: Thread pool created using 64 threads
x265 [info]: Slices                              : 1
x265 [info]: frame threads / pool features       : 6 / wpp(51 rows)
x265 [info]: Coding QT: max CU size, min CU size : 64 / 8
x265 [info]: Residual QT: max TU size, max depth : 32 / 1 inter / 1 intra
x265 [info]: ME / range / subpel / merge         : hex / 57 / 2 / 3
x265 [info]: Keyframe min / max / scenecut / bias  : 24 / 250 / 40 / 5.00 
x265 [info]: Lookahead / bframes / badapt        : 20 / 4 / 2
x265 [info]: b-pyramid / weightp / weightb       : 1 / 1 / 0
x265 [info]: References / ref-limit  cu / depth  : 3 / off / on
x265 [info]: AQ: mode / str / qg-size / cu-tree  : 2 / 1.0 / 32

...

Rendering frames:   0%|          | 0/351 [00:00<?, ?it/s]

🎬 Assembling media file (Streaming Mode + HEVC)...


Saving MP4:   0%|          | 0/351 [00:00<?, ?it/s]

x265 [info]: HEVC encoder version 3.5+1-f0c1022b6
x265 [info]: build info [Linux][GCC 8.3.0][64 bit] 8bit+10bit+12bit
x265 [info]: using cpu capabilities: MMX2 SSE2Fast LZCNT SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
x265 [info]: Main profile, Level-6 (Main tier)
x265 [info]: Thread pool created using 64 threads
x265 [info]: Thread pool created using 64 threads
x265 [info]: Slices                              : 1
x265 [info]: frame threads / pool features       : 6 / wpp(51 rows)
x265 [info]: Coding QT: max CU size, min CU size : 64 / 8
x265 [info]: Residual QT: max TU size, max depth : 32 / 1 inter / 1 intra
x265 [info]: ME / range / subpel / merge         : hex / 57 / 2 / 3
x265 [info]: Keyframe min / max / scenecut / bias  : 24 / 250 / 40 / 5.00 
x265 [info]: Lookahead / bframes / badapt        : 20 / 4 / 2
x265 [info]: b-pyramid / weightp / weightb       : 1 / 1 / 0
x265 [info]: References / ref-limit  cu / depth  : 3 / off / on
x265 [info]: AQ: mode / str / qg-size / cu-tree  : 2 / 1.0 / 32

✅ Success! Render saved to: ../media/demo-arena-2-basic.mp4
🧹 Cleaning up temporary frames...


#### 📺 Side-by-Side Trajectory Visualization

The cell below uses custom HTML and JavaScript to display both generated simulations simultaneously. This layout facilitates a direct, frame-by-frame comparison of how the extended primitive set optimizes the path layout against the standard baseline. 

*Note: Double-click the cell below if you want to inspect the HTML grid or the synchronization script properties.*


In [26]:
%%html
<!-- Container for side-by-side video rendering using CSS Flexbox -->
<div style="display: flex; gap: 20px; justify-content: center; align-items: center; width: 100%;">
    <!-- Left Column: Extended Control Set -->
    <div style="text-align: center; flex: 1;">
        <h4 style="margin-bottom: 10px;">🚀 Extended Control Set (Efficient)</h4>
        <video id="video-extended" src="../media/demo-arena-2-extended.mp4" controls width="100%" muted></video>
    </div>
    <!-- Right Column: Baseline Control Set -->
    <div style="text-align: center; flex: 1;">
        <h4 style="margin-bottom: 10px;">📉 Baseline Control Set (Standard)</h4>
        <video id="video-baseline" src="../media/demo-arena-2-basic.mp4" controls width="100%" muted></video>
    </div>
</div>

<script>
(function() {
    // Retrieve DOM elements for both video players
    const vid1 = document.getElementById('video-extended');
    const vid2 = document.getElementById('video-baseline');

    if (vid1 && vid2) {
        // Synchronize playback events triggered from the Extended video player
        vid1.onplay = () => vid2.play();
        vid1.onpause = () => vid2.pause();
        vid1.onseeking = () => { vid2.currentTime = vid1.currentTime; };

        // Synchronize playback events triggered from the Baseline video player
        vid2.onplay = () => vid1.play();
        vid2.onpause = () => vid1.pause();
        vid2.onseeking = () => { vid1.currentTime = vid2.currentTime; };
    }
})();
</script>

# Empty Map Task with Big Agent

In [51]:
task_map = Map()
task_map.read_from_movingai_file("../maps/empty_64_64.map")

R = 3  # Ego-robot radius
environment = DynamicEnvironment(task_map, R)
environment.load_obstacles_from_dir("../dynamic_obstacles/empty_64_64_circle/dynamic_obstacle_*.txt", env_cs, max_items=80)
environment.compile_safe_intervals()

start = DiscreteState(9, 9, 0)
goal = DiscreteState(41, 23, 0)
search_space = SIPPSearchSpace(start, 0.0, goal, extended_cs, environment, R=0, A=0, position_only=True)

success, path, steps, cost, ast = astar(search_space)
success, steps, cost

(True, 1990, np.float64(610.986255))

In [52]:
robot = DynamicObstacle(extended_cs).set_parametres(R, start.i, start.j, start.theta, path)

In [55]:
render_simulation_parallel(
    output_file="../media/demo-empty-1.mp4",
    task_map=task_map,
    obstacle_set=environment.get_dynamic_obstacles_set(),
    robot=robot,
    max_time=620.0,
    dt=1.0,
    fps=40,
    show_path=True,
    obs_line_width=0.4,
    fast_draw=True,
    dpi=200,
    scale=2,
    workers=50
)

🚀 Starting parallel rendering...
📊 Total frames: 621 | CPU Workers: 50


Rendering frames:   0%|          | 0/621 [00:00<?, ?it/s]

🎬 Assembling media file (Streaming Mode + HEVC)...


Saving MP4:   0%|          | 0/621 [00:00<?, ?it/s]

x265 [info]: HEVC encoder version 3.5+1-f0c1022b6
x265 [info]: build info [Linux][GCC 8.3.0][64 bit] 8bit+10bit+12bit
x265 [info]: using cpu capabilities: MMX2 SSE2Fast LZCNT SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
x265 [info]: Main profile, Level-5 (Main tier)
x265 [info]: Thread pool created using 64 threads
x265 [info]: Thread pool created using 64 threads
x265 [info]: Slices                              : 1
x265 [info]: frame threads / pool features       : 6 / wpp(41 rows)
x265 [info]: Coding QT: max CU size, min CU size : 64 / 8
x265 [info]: Residual QT: max TU size, max depth : 32 / 1 inter / 1 intra
x265 [info]: ME / range / subpel / merge         : hex / 57 / 2 / 3
x265 [info]: Keyframe min / max / scenecut / bias  : 25 / 250 / 40 / 5.00 
x265 [info]: Lookahead / bframes / badapt        : 20 / 4 / 2
x265 [info]: b-pyramid / weightp / weightb       : 1 / 1 / 0
x265 [info]: References / ref-limit  cu / depth  : 3 / off / on
x265 [info]: AQ: mode / str / qg-size / cu-tree  : 2 / 1.0 / 32

✅ Success! Render saved to: ../media/demo-empty-1.mp4
🧹 Cleaning up temporary frames...


# Empty Map for 2^3

In [57]:
cs_3 = ControlSet().load_primitives("../data/primitives_2k_3.txt")

In [58]:
start = DiscreteState(9, 9, 0)
goal = DiscreteState(36, 35, 0)
search_space = SIPPSearchSpace(start, 0.0, goal, cs_3, environment, R=0, A=0, position_only=True)

success, path, steps, cost, ast = astar(search_space)
success, steps, cost

(True, 4873, np.float64(1221.913086))

In [60]:
robot = DynamicObstacle(cs_3).set_parametres(R, start.i, start.j, start.theta, path)
render_simulation_parallel(
    output_file="../media/demo-empty-2.mp4",
    task_map=task_map,
    obstacle_set=environment.get_dynamic_obstacles_set(),
    robot=robot,
    max_time=1230.0,
    dt=1.0,
    fps=40,
    show_path=True,
    obs_line_width=0.4,
    fast_draw=True,
    dpi=200,
    scale=2,
    workers=60
)

🚀 Starting parallel rendering...
📊 Total frames: 1231 | CPU Workers: 60


Rendering frames:   0%|          | 0/1231 [00:00<?, ?it/s]

🎬 Assembling media file (Streaming Mode + HEVC)...


Saving MP4:   0%|          | 0/1231 [00:00<?, ?it/s]

x265 [info]: HEVC encoder version 3.5+1-f0c1022b6
x265 [info]: build info [Linux][GCC 8.3.0][64 bit] 8bit+10bit+12bit
x265 [info]: using cpu capabilities: MMX2 SSE2Fast LZCNT SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
x265 [info]: Main profile, Level-5 (Main tier)
x265 [info]: Thread pool created using 64 threads
x265 [info]: Thread pool created using 64 threads
x265 [info]: Slices                              : 1
x265 [info]: frame threads / pool features       : 6 / wpp(41 rows)
x265 [info]: Coding QT: max CU size, min CU size : 64 / 8
x265 [info]: Residual QT: max TU size, max depth : 32 / 1 inter / 1 intra
x265 [info]: ME / range / subpel / merge         : hex / 57 / 2 / 3
x265 [info]: Keyframe min / max / scenecut / bias  : 25 / 250 / 40 / 5.00 
x265 [info]: Lookahead / bframes / badapt        : 20 / 4 / 2
x265 [info]: b-pyramid / weightp / weightb       : 1 / 1 / 0
x265 [info]: References / ref-limit  cu / depth  : 3 / off / on
x265 [info]: AQ: mode / str / qg-size / cu-tree  : 2 / 1.0 / 32

✅ Success! Render saved to: ../media/demo-empty-2.mp4
🧹 Cleaning up temporary frames...


# Huge Denver Map

In [66]:
task_map = Map()
task_map.read_from_movingai_file("../maps/Denver_1_256.map")

R = 2  # Ego-robot radius
environment = DynamicEnvironment(task_map, R)
environment.load_obstacles_from_dir("../dynamic_obstacles/Denver_1_256_random/dynamic_obstacle_*.txt", env_cs, max_items=320)
environment.compile_safe_intervals()

start = DiscreteState(250, 80, 0)
goal = DiscreteState(15, 225, -4)
search_space = SIPPSearchSpace(start, 0.0, goal, env_cs, environment, R=0, A=0, position_only=False)

success, path, steps, cost, ast = astar(search_space, max_time_sec=2000.0)
success, steps, cost

(True, 360743, np.float64(4156.352477))

In [67]:
robot = DynamicObstacle(env_cs).set_parametres(R, start.i, start.j, start.theta, path)
render_simulation_parallel(
    output_file="../media/demo-denver-1.mp4",
    task_map=task_map,
    obstacle_set=environment.get_dynamic_obstacles_set(),
    robot=robot,
    max_time=4200.0,
    dt=2.0,
    fps=24,
    show_path=True,
    obs_line_width=0.4,
    fast_draw=True,
    dpi=100,
    scale=2,
    workers=100
)

🚀 Starting parallel rendering...
📊 Total frames: 2101 | CPU Workers: 100


Rendering frames:   0%|          | 0/2101 [00:00<?, ?it/s]

🎬 Assembling media file (Streaming Mode + HEVC)...


Saving MP4:   0%|          | 0/2101 [00:00<?, ?it/s]

x265 [info]: HEVC encoder version 3.5+1-f0c1022b6
x265 [info]: build info [Linux][GCC 8.3.0][64 bit] 8bit+10bit+12bit
x265 [info]: using cpu capabilities: MMX2 SSE2Fast LZCNT SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
x265 [info]: Main profile, Level-4 (Main tier)
x265 [info]: Thread pool created using 64 threads
x265 [info]: Thread pool created using 64 threads
x265 [info]: Slices                              : 1
x265 [info]: frame threads / pool features       : 5 / wpp(21 rows)
x265 [info]: Coding QT: max CU size, min CU size : 64 / 8
x265 [info]: Residual QT: max TU size, max depth : 32 / 1 inter / 1 intra
x265 [info]: ME / range / subpel / merge         : hex / 57 / 2 / 3
x265 [info]: Keyframe min / max / scenecut / bias  : 24 / 250 / 40 / 5.00 
x265 [info]: Lookahead / bframes / badapt        : 20 / 4 / 2
x265 [info]: b-pyramid / weightp / weightb       : 1 / 1 / 0
x265 [info]: References / ref-limit  cu / depth  : 3 / off / on
x265 [info]: AQ: mode / str / qg-size / cu-tree  : 2 / 1.0 / 32

✅ Success! Render saved to: ../media/demo-denver-1.mp4
🧹 Cleaning up temporary frames...


In [68]:
print(path)

[0, 0, np.float64(-6.667452999999995), 24, 99, 61, 101, 59, 91, 15, 70, np.float64(-8.53021799999999), 55, 75, 70, np.float64(-138.8748559999999), 51, 78, 75, 70, 31, 55, 59, 67, 47, 15, 70, 35, 104, np.float64(-12.176738000000569), 31, 51, 62, 62, 86, np.float64(-169.48635699999977), 7, np.float64(-4.73416700000007), 107, np.float64(-1.4672679999998763), 50, np.float64(-74.29263200000014), 61, 69, np.float64(-363.3266160000003), 34, np.float64(-0.20922499999960564), 99, np.float64(-18.475304999999935), 101, np.float64(-5.111641999999847), 91]
